# 01 · GSE65391 · RNA_array · metadata

Reads the sample characteristics from the series matrix. Writes `data/run_artifacts/GSE65391/metadata.rds`.

GEO stores each characteristic as `key: value`. `Not Applicable`, `Data Not Available`, `N/A` and `Unk.`
become missing values.

Derived fields:
- **stage**: SLEDAI category per visit: none 0, mild 1–5, moderate 6–10, high 11–19, very high ≥ 20.
- **nephritis_class**: the field arrives as two values joined by `;` (for example
  `Data Not Available;Prolif`); the class is the second value.
- **first_visit**: TRUE for the lowest visit number of each child.

In [1]:
source("../src/paths.R")
suppressMessages({library(GEOquery); library(Biobase)})
pd <- pData(suppressMessages(getGEO(filename = raw("GSE65391", "GSE65391_series_matrix.txt.gz"), getGPL = FALSE)))
c(samples = nrow(pd), characteristic_fields = length(grep(":ch1$", names(pd))))

samples characteristic_fields 
                  996                    87

In [2]:
ch <- function(key) {
  v <- as.character(pd[[paste0(key, ":ch1")]])
  v[v %in% c("Not Applicable", "Data Not Available", "NA", "N/A", "Unk.", "")] <- NA
  v
}
num <- function(key) suppressWarnings(as.numeric(ch(key)))

meta <- data.frame(
  sample = rownames(pd), subject = ch("subject"),
  disease = factor(ch("disease state"), levels = c("Healthy", "SLE")),
  visit = as.integer(ch("visit")), batch = ch("batch"), set = ch("set"),
  age = num("age"), sex = ch("gender"), race = ch("race"),
  sledai = num("sledai"),
  nephritis_class = sub(".*;", "", ch("nephritis_class")),
  mdg = ch("mdg"),
  c3 = num("c3"), c4 = num("c4"), ds_dna = num("ds_dna"),
  wbc = num("wbc"), neutrophil_count = num("neutrophil_count"),
  lymphocyte_count = num("lymphocyte_count"), platelet_count = num("platelet_count"),
  oral_steroids = num("oral_steroids_category"), iv_steroids = num("steroid_iv_category"),
  mycophenolate = num("mycophenolate_category"), hydroxychloroquine = num("hydroxychloroquine_category"),
  cyclophosphamide = num("cyclophosphamide_category"),
  stringsAsFactors = FALSE, row.names = rownames(pd))
meta$nephritis_class[meta$nephritis_class %in% c("Not Applicable", "Data Not Available")] <- NA
meta$stage <- cut(meta$sledai, c(-Inf, 0, 5, 10, 19, Inf),
                  labels = c("none", "mild", "moderate", "high", "very high"), ordered_result = TRUE)
# first visit: the lowest recorded visit number of each child; a sample with no visit number is not a first visit
meta$first_visit <- !is.na(meta$visit) &
  meta$visit == ave(meta$visit, meta$subject, FUN = function(v) if (all(is.na(v))) NA else min(v, na.rm = TRUE))
addmargins(table(disease = meta$disease, batch = meta$batch))

,1,2,Sum
Healthy,32,40,72
SLE,806,118,924
Sum,838,158,996


In [3]:
c(SLE_children = length(unique(meta$subject[meta$disease == "SLE"])),
  healthy_children = length(unique(meta$subject[meta$disease == "Healthy"])),
  SLE_samples = sum(meta$disease == "SLE"))
table(stage = meta$stage, useNA = "ifany")
table(nephritis_class = meta$nephritis_class, useNA = "ifany")
table(sex = meta$sex, disease = meta$disease)

SLE_children healthy_children      SLE_samples 
             158               46              924

stage
     none      mild  moderate      high very high      <NA> 
       93       390       277       127        37        72 

nephritis_class
      Membr       Mesan        NoLN Proli+Membr      Prolif        <NA> 
         42           8         498           3         238         207 

   disease
sex Healthy SLE
  F      57 817
  M      15 107

**Result.** 158 children with SLE contribute 924 samples; 46 healthy children contribute 72, of
which 24 are technical replicates.

**Check.** Samples with no visit number in GEO.

In [4]:
meta[is.na(meta$visit), c("subject", "disease", "sledai")]
table(visits_of_that_child = meta$subject %in% meta$subject[is.na(meta$visit)] & !is.na(meta$visit))

,subject,disease,sledai
,<chr>,<fct>,<dbl>
GSM1594705,SLE-222,SLE,4


visits_of_that_child
FALSE 
  996 

**Result.** One sample, GSM1594705 (child SLE-222), has no visit number, and it is that child's only
sample. It is not counted as a first visit, so first visits number 157, not 158.

First visits of children with SLE: these are the samples WGCNA uses (one sample per child).

In [5]:
fv <- meta[meta$disease == "SLE" & meta$first_visit & meta$set != "Technical_Replicate", ]
c(children = nrow(fv), unique_subjects = length(unique(fv$subject)))
summary(fv[, c("age", "sledai", "c3", "c4", "ds_dna", "neutrophil_count", "lymphocyte_count")])

children unique_subjects 
            157             157

      age            sledai             c3              c4       
 Min.   : 6.00   Min.   : 0.000   Min.   : 20.0   Min.   : 0.00  
 1st Qu.:12.31   1st Qu.: 4.000   1st Qu.: 65.0   1st Qu.: 7.00  
 Median :14.38   Median : 6.000   Median : 96.0   Median :14.50  
 Mean   :14.05   Mean   : 8.567   Mean   : 94.1   Mean   :16.62  
 3rd Qu.:16.22   3rd Qu.:12.000   3rd Qu.:123.0   3rd Qu.:25.00  
 Max.   :18.45   Max.   :35.000   Max.   :180.0   Max.   :65.00  
                                  NA's   :8       NA's   :9      
     ds_dna         neutrophil_count lymphocyte_count
 Min.   :   0.049   Min.   : 0.600   Min.   : 0.290  
 1st Qu.:  10.600   1st Qu.: 2.410   1st Qu.: 0.980  
 Median :  26.800   Median : 3.765   Median : 1.370  
 Mean   : 191.273   Mean   : 4.113   Mean   : 1.958  
 3rd Qu.:  92.350   3rd Qu.: 5.305   3rd Qu.: 2.000  
 Max.   :5531.000   Max.   :10.640   Max.   :57.500  
 NA's   :67         NA's   :19       NA's   :20      

In [6]:
saveRDS(meta, art("GSE65391", "metadata.rds"))